# Advanced layer types -- Image Classification

## 0. Import packages and modules

In [ ]:
import os

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from tqdm import tqdm

import pathlib

import sklearn
import torch
from torch import nn

from torchinfo import summary
#import torchmetrics

print(sklearn.__version__)
print(torch.__version__)

In [ ]:
# print GPU info

print(f"CUDA available: {torch.cuda.is_available()}")

print(f"Num GPUs Available: {torch.cuda.device_count()}")

if torch.cuda.is_available():
    device = torch.device("cuda")
    # Requires nvidia-ml-py or equivalent
    # print(f"{torch.cuda.get_device_name(device)}: temp: {torch.cuda.temperature(device)} °C, power draw: {torch.cuda.power_draw()} mW, clock rate: {torch.cuda.clock_rate()} MHz")
else:
    device = torch.device("cpu")

## 1. Formulate / Outline the problem: Image classification

In [ ]:
DATA_FOLDER = pathlib.Path(f"/scratch/project_465002387/data/DollarStreet10") # change to location where you stored the data

train_images = np.load(DATA_FOLDER / 'train_images.npy')
val_images = np.load(DATA_FOLDER / 'test_images.npy')
train_labels = np.load(DATA_FOLDER / 'train_labels.npy')
val_labels = np.load(DATA_FOLDER / 'test_labels.npy')

## 2. Identify inputs and outputs

In [ ]:
train_images.shape

In [ ]:
train_images.min(), train_images.max()

In [ ]:
train_labels.shape

In [ ]:
train_labels.min(), train_labels.max()

## 3. Prepare data

In [ ]:
class DollarStreetDataset(torch.utils.data.Dataset):
    def __init__(self, root, train=True):
        prefix = "test"
        if train == True:
            prefix = "train"
        self.images = np.load(root / f'{prefix}_images.npy') / 255.
        self.labels = np.load(root / f'{prefix}_labels.npy')
        
    def __getitem__(self, idx):
        x = torch.permute(torch.tensor(self.images[idx], dtype=torch.float), (2, 0, 1))
        y = torch.tensor(self.labels[idx], dtype=torch.long)
        return x, y
        
    def __len__(self):
        return self.images.shape[0]

train_ds = DollarStreetDataset(root=DATA_FOLDER, train=True)
val_ds = DollarStreetDataset(root=DATA_FOLDER, train=False)

## 4. Build a pretrained model

In [ ]:
class DollarStreetModelSmall(nn.Module):
    def __init__(self):
        super().__init__()
        self.s = nn.Sequential(
            nn.Conv2d(3, 50, 3),
            nn.ReLU(),
            nn.Conv2d(50, 50, 3),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(50*60*60, 10),
        )
    def forward(self, x):
        return self.s(x)

model = DollarStreetModelSmall()

In [ ]:
summary(model, input_size=(1, 3, 64, 64))

In [ ]:
class DollarStreetModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.s = nn.Sequential(
            nn.Conv2d(3, 50, 3),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(50, 50, 3),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(14*14*50, 50),
            nn.ReLU(),
            nn.Linear(50, 10),
        )
    def forward(self, x):
        return self.s(x)

model = DollarStreetModel()
summary(model, input_size=(1, 3, 64, 64))

## 5. Choose a loss function and optimizer

In [ ]:
optim = torch.optim.Adam
loss_fn = torch.nn.CrossEntropyLoss

## 6. Train the model

In [ ]:
def train(dl, model, loss_fn, optimizer, device=torch.device("cuda:0")):
    model = model.to(device)
    model.train()

    avg_loss = 0.0
    avg_acc = 0.0
    
    for step, (x, y) in enumerate(tqdm(dl)):
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        pred = model(x)
        loss_val = loss_fn(pred, y)
        loss_val.backward()
        optimizer.step()

        avg_loss += loss_val.item()
        avg_acc += (pred.argmax(1) == y).sum().item()
    avg_loss /= len(dl)
    avg_acc /= len(dl.dataset)
        
    return avg_loss, avg_acc

def test(dl, model, loss_fn, device=torch.device("cuda:0")):
    model = model.to(device)
    model.eval()
    
    avg_loss = 0.0
    avg_acc = 0.0
    with torch.no_grad():
        for step, (x, y) in enumerate(dl):
            x, y = x.to(device), y.to(device)
            pred = model(x)
            avg_loss += loss_fn(pred, y).item()
            avg_acc += (pred.argmax(1) == y).type(torch.float).sum().item()
    avg_loss /= len(dl)
    avg_acc /= (len(dl.dataset))
    print(f"Validation accuracy: {(100 * avg_acc):>0.1f}%, Avg loss: {avg_loss:>8f}")
    return avg_loss, avg_acc

def fit(model, optimizer, loss, train_ds, val_ds, batch_size=32, learning_rate=0.001, num_epochs=20):

    # instantiate optimizer and loss_fn
    loss_fn = loss()
    optim = optimizer(model.parameters(), lr=learning_rate)

    train_dl = torch.utils.data.DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_dl = torch.utils.data.DataLoader(val_ds, batch_size=1, shuffle=False)

    history = {"loss": [], "val_loss": [], "accuracy": [], "val_accuracy": []}
    for epoch in range(num_epochs):
        train_loss, train_acc = train(train_dl, model, loss_fn, optim, device=device)
        val_loss, val_acc = test(val_dl, model, loss_fn, device=device)

        for k, v in [("loss", train_loss), ("val_loss", val_loss), ("accuracy", train_acc), ("val_accuracy", val_acc)]:
            history[k].append(v)
    
    return history

In [ ]:
history = fit(model, optim, loss_fn, train_ds, val_ds, learning_rate=0.0001)

## 7. Perform a Prediction/Classification

Here we skip performing a prediction, and continue to measuring the performance.

## 8. Measure performance

In [ ]:
def plot_history(history, metrics):
    """
    Plot the training history

    Args:
        history(dict): dictionary containing metrics over the course of training
        metrics(str, list): Metric or a list of metrics to plot
    """
    history_df = pd.DataFrame.from_dict(history)
    sns.lineplot(data=history_df[metrics])
    plt.xlabel("epochs")
    plt.ylabel("metric")


In [ ]:
plot_history(history, ['accuracy', 'val_accuracy'])

In [ ]:
plot_history(history, ['loss', 'val_loss'])

**Comparison with a network with only dense layers**

In [ ]:
class DenseModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.s = nn.Sequential(
            nn.Flatten(),
            nn.Linear(12288, 50),
            nn.ReLU(),
            nn.Linear(50, 50),
            nn.ReLU(),
            nn.Linear(50, 10),
        )
    def forward(self, x):
        return self.s(x)

dense_model = DenseModel()
summary(dense_model, input_size=(1, 3, 64, 64))

In [ ]:
dense_history = fit(dense_model, optim, loss_fn, train_ds, val_ds, learning_rate=0.0001)

In [ ]:
plot_history(dense_history, ['accuracy', 'val_accuracy'])

In [ ]:
plot_history(dense_history, ['loss', 'val_loss'])

## 9. Refine the model

In [ ]:
class ModelDropout(nn.Module):
    def __init__(self):
        super().__init__()
        self.s = nn.Sequential(
            nn.Conv2d(3, 50, 3),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(0.5),  # This is new!
            nn.Conv2d(50, 50, 3),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(0.5),  # This is new!
            nn.Conv2d(50, 50, 3),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(0.5),  # This is new!
            nn.Flatten(),
            nn.Linear(1800, 50),
            nn.ReLU(),
            nn.Linear(50, 10),
        )
    def forward(self, x):
        return self.s(x)

dropout_model = ModelDropout()
summary(dropout_model, input_size=(1, 3, 64, 64))

In [ ]:
dropout_history = fit(dropout_model, optim, loss_fn, train_ds, val_ds, learning_rate=0.0001)

In [ ]:
plot_history(dropout_history, ['loss', 'val_loss'])

In [ ]:
plot_history(dropout_history, ['accuracy', 'val_accuracy'])

In [ ]:
class HPModel(nn.Module):
    def __init__(self, dropout_rate, n_layers):
        super().__init__()
        self.in_layer = nn.Sequential(nn.Conv2d(3, 50, 3), nn.ReLU(), nn.MaxPool2d(2))
        self.hidden_layers = nn.Sequential(*[nn.Sequential(nn.Conv2d(50, 50, 3), nn.ReLU(), nn.MaxPool2d(2)) for n in range(n_layers-1)])
        n_features_after_flatten = 64
        for n in range(n_layers):
            n_features_after_flatten = (n_features_after_flatten - 2)//2

        n_features_after_flatten = n_features_after_flatten ** 2 * 50
        self.head = nn.Sequential(nn.Dropout(dropout_rate), nn.Flatten(), nn.Linear(n_features_after_flatten , 50), nn.ReLU(), nn.Linear(50, 10))

    def forward(self, x):
        x = self.in_layer(x)
        x = self.hidden_layers(x)
        x = self.head(x)
        return x

In [ ]:
n_layers_grid = [1, 2]
dropout_rate_grid = np.linspace(.2, .8, 3)

best = None
best_loss = float("inf")
best_model = None
histories = {}
trial = 0

for n_layers in n_layers_grid:
    for dropout_rate in dropout_rate_grid:

        history = fit(HPModel(dropout_rate, n_layers), optim, loss_fn, train_ds, val_ds, learning_rate=0.0001)
        histories[n_layers, dropout_rate] = history

        val_loss = min(history["loss"])
        if val_loss < best_loss:
            best_loss = val_loss
            best = (n_layers, float(dropout_rate),)
            best_model = model
            
        print(
            f"trial {trial}: params: {n_layers=}, {dropout_rate}; "
            f"best: {best_loss} at (n_layers, dropout_rate)={best}"
        )
        trial += 1


In [ ]:
print(f"best val loss: {best_loss:.4f} at (n_layers, dropout_rate)={best}")
print("trials executed:", trial)

## 10. Share model

In [ ]:
torch.save(best_model.state_dict(), "image_classifcation.pth")

In [ ]:
print(len(train_ds))
print(len(val_ds))